# Introduction to Data Ethics and Privacy

This notebook introduces fundamental concepts in data ethics and privacy, essential knowledge for any data scientist. We'll explore ethical considerations when working with data, privacy regulations, and best practices for responsible data science.

## 1. Import Essential Libraries

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
plt.style.use('ggplot')
sns.set(style="whitegrid")

# For privacy-preserving techniques
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# For displaying sample data
from IPython.display import display, HTML

print("Libraries imported successfully!")

## 2. Key Ethical Concepts in Data Science

Data ethics refers to the moral principles and practices governing data collection, usage, and sharing. Key concepts include:

1. **Fairness**: Ensuring algorithms don't discriminate against certain groups
2. **Transparency**: Making data collection and usage processes clear to users
3. **Privacy**: Protecting sensitive information from unauthorized access
4. **Consent**: Obtaining proper permission from individuals for data use
5. **Accountability**: Taking responsibility for the consequences of data analysis

## 3. Privacy Regulations and Frameworks

Several key regulations govern data privacy globally:

In [ ]:
# Create a DataFrame of major privacy regulations
regulations = pd.DataFrame({
    'Regulation': ['GDPR', 'CCPA', 'HIPAA', 'PIPEDA', 'LGPD'],
    'Full Name': [
        'General Data Protection Regulation',
        'California Consumer Privacy Act',
        'Health Insurance Portability and Accountability Act',
        'Personal Information Protection and Electronic Documents Act',
        'Lei Geral de Proteção de Dados'
    ],
    'Region': ['European Union', 'California (USA)', 'United States', 'Canada', 'Brazil'],
    'Year Effective': [2018, 2020, 1996, 2000, 2020]
})

# Display the regulations table with styling
display(regulations.style.set_caption("Major Data Privacy Regulations"))

# Create a bar chart showing when regulations were implemented
plt.figure(figsize=(12, 5))
sns.barplot(x='Regulation', y='Year Effective', data=regulations)
plt.title('Timeline of Major Privacy Regulations')
plt.ylabel('Year Implemented')
plt.xlabel('Regulation')
plt.show()

### GDPR Key Principles

The General Data Protection Regulation (GDPR) introduced several important principles that have influenced global data privacy standards:

1. **Lawfulness, fairness and transparency**: Data processing must be legal, fair, and transparent
2. **Purpose limitation**: Collect data for specific, explicit purposes
3. **Data minimization**: Only collect what's necessary
4. **Accuracy**: Ensure data is accurate and up-to-date
5. **Storage limitation**: Don't keep data longer than needed
6. **Integrity and confidentiality**: Ensure appropriate security measures
7. **Accountability**: Be able to demonstrate compliance

## 4. Creating a Synthetic Dataset for Privacy Demonstration

Let's create a sample dataset that resembles sensitive personal information to demonstrate privacy concepts:

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate a synthetic dataset with personal information
n_samples = 100

# Create sample data
names = [f"Person_{i}" for i in range(1, n_samples+1)]
ages = np.random.randint(18, 85, n_samples)
salaries = np.random.normal(60000, 15000, n_samples).astype(int)
emails = [f"person_{i}@example.com" for i in range(1, n_samples+1)]
zip_codes = np.random.randint(10000, 99999, n_samples)
medical_conditions = np.random.choice(
    ["None", "Diabetes", "Hypertension", "Asthma", "Heart Disease"], 
    n_samples
)
credit_scores = np.random.randint(300, 850, n_samples)

# Create DataFrame
personal_data = pd.DataFrame({
    'Name': names,
    'Age': ages,
    'Salary': salaries,
    'Email': emails,
    'ZIP': zip_codes,
    'MedicalCondition': medical_conditions,
    'CreditScore': credit_scores
})

# Display first few rows
display(personal_data.head().style.set_caption("Synthetic Personal Data (Sample)"))

# Show summary statistics
display(personal_data.describe().style.set_caption("Summary Statistics"))

## 5. Identifying Sensitive Data

Not all data is equally sensitive. Different types of data require different levels of protection:

### Types of Sensitive Data:

1. **Personally Identifiable Information (PII)**: Names, addresses, phone numbers, etc.
2. **Protected Health Information (PHI)**: Medical records, health insurance info
3. **Financial Data**: Credit card numbers, bank accounts
4. **Biometric Data**: Fingerprints, facial recognition data
5. **Behavioral Data**: Browsing history, app usage patterns

In [ ]:
# Categorize columns by sensitivity level
sensitivity_levels = pd.DataFrame({
    'Data Field': personal_data.columns.tolist(),
    'Sensitivity': ['Medium', 'Medium', 'High', 'Medium', 'Medium', 'Very High', 'High'],
    'Category': ['PII', 'PII', 'Financial', 'PII', 'PII', 'PHI', 'Financial'],
    'Risk if Exposed': ['Identity can be determined', 
                        'Age discrimination possible',
                        'Financial targeting, discrimination',
                        'Phishing, spam',
                        'Location can be determined',
                        'Health discrimination, stigmatization',
                        'Financial targeting']
})

# Display sensitivity table with conditional formatting
def highlight_sensitivity(val):
    color_map = {
        'Low': 'lightgreen',
        'Medium': 'khaki',
        'High': 'salmon',
        'Very High': 'lightcoral'
    }
    if val in color_map:
        return f'background-color: {color_map[val]}'
    return ''

display(sensitivity_levels.style.applymap(highlight_sensitivity, subset=['Sensitivity'])
       .set_caption("Data Sensitivity Analysis"))

## 6. De-identification Techniques

De-identification is the process of removing or obscuring personal identifiers from data. Let's explore common techniques:

In [ ]:
# Create a copy of our data to apply de-identification techniques
anonymized_data = personal_data.copy()

# 1. Suppression - completely removing certain fields
anonymized_data_1 = anonymized_data.drop(columns=['Name', 'Email'])

# 2. Generalization - reducing precision of data
anonymized_data_2 = anonymized_data.copy()
# Group ages into ranges
anonymized_data_2['Age'] = pd.cut(anonymized_data_2['Age'], 
                                 bins=[0, 25, 35, 50, 65, 100],
                                 labels=['18-25', '26-35', '36-50', '51-65', '65+'])
# Convert ZIP to first 3 digits only
anonymized_data_2['ZIP'] = anonymized_data_2['ZIP'].astype(str).str[:3] + 'XX'

# 3. Perturbation - adding noise to numerical data
anonymized_data_3 = anonymized_data.copy()
noise_factor = 0.05  # 5% noise
anonymized_data_3['Salary'] = anonymized_data_3['Salary'] * (1 + np.random.normal(0, noise_factor, n_samples))
anonymized_data_3['Salary'] = anonymized_data_3['Salary'].astype(int)
anonymized_data_3['CreditScore'] = anonymized_data_3['CreditScore'] + np.random.randint(-20, 21, n_samples)
# Ensure credit scores stay in valid range
anonymized_data_3['CreditScore'] = np.clip(anonymized_data_3['CreditScore'], 300, 850)

# Display results
print("Original Data:")
display(personal_data.head(3))

print("\nTechnique 1: Suppression")
display(anonymized_data_1.head(3))

print("\nTechnique 2: Generalization")
display(anonymized_data_2.head(3))

print("\nTechnique 3: Perturbation (Adding Noise)")
display(anonymized_data_3.head(3))

## 7. K-Anonymity: A Common Privacy Standard

K-anonymity is a property of an anonymized dataset where each record is indistinguishable from at least k-1 other records on the potentially identifying attributes. This ensures that individuals cannot be uniquely identified in the dataset.

In [ ]:
# Implement a basic k-anonymity check function
def check_k_anonymity(data, quasi_identifiers, k=2):
    """
    Check if the dataset satisfies k-anonymity for the given quasi-identifiers.
    
    Parameters:
    data (pandas.DataFrame): Dataset to check
    quasi_identifiers (list): List of column names that act as quasi-identifiers
    k (int): Minimum number of records that should share the same values for quasi-identifiers
    
    Returns:
    bool: True if k-anonymity is satisfied, False otherwise
    """
    # Group by the combination of quasi-identifiers and count occurrences
    group_counts = data.groupby(quasi_identifiers).size()
    
    # Check if all groups have at least k records
    is_k_anonymous = all(count >= k for count in group_counts)
    
    # Get minimum anonymity level
    min_anonymity = group_counts.min() if len(group_counts) > 0 else 0
    
    return is_k_anonymous, min_anonymity, group_counts

# Define quasi-identifiers
quasi_identifiers = ['Age', 'ZIP', 'MedicalCondition']

# Check k-anonymity on original data
k_anon_original, min_anon_original, groups_original = check_k_anonymity(
    personal_data, quasi_identifiers, k=2
)

# Check k-anonymity on generalized data
k_anon_generalized, min_anon_generalized, groups_generalized = check_k_anonymity(
    anonymized_data_2, quasi_identifiers, k=2
)

print(f"Original data satisfies 2-anonymity: {k_anon_original}")
print(f"Minimum anonymity level in original data: {min_anon_original}")
print("\nTop 5 most common combinations in original data:")
display(groups_original.sort_values(ascending=False).head(5))

print(f"\nGeneralized data satisfies 2-anonymity: {k_anon_generalized}")
print(f"Minimum anonymity level in generalized data: {min_anon_generalized}")
print("\nTop 5 most common combinations in generalized data:")
display(groups_generalized.sort_values(ascending=False).head(5))

# Calculate frequency of each k value in original vs generalized data
original_k_freq = groups_original.value_counts().sort_index()
generalized_k_freq = groups_generalized.value_counts().sort_index()

# Plot comparison
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
original_k_freq.plot.bar(ax=ax[0], color='salmon')
ax[0].set_title('Original Data: Frequency of Group Sizes')
ax[0].set_xlabel('Records per Group (k)')
ax[0].set_ylabel('Number of Groups')

generalized_k_freq.plot.bar(ax=ax[1], color='lightgreen')
ax[1].set_title('Generalized Data: Frequency of Group Sizes')
ax[1].set_xlabel('Records per Group (k)')
ax[1].set_ylabel('Number of Groups')

plt.tight_layout()
plt.show()

## 8. Privacy-Preserving Analysis Techniques

When working with sensitive data, consider these privacy-preserving analysis techniques:

1. **Aggregation**: Work with summary statistics instead of raw data
2. **Differential Privacy**: Add controlled noise to protect individuals
3. **Federated Learning**: Train models without centralizing data
4. **Secure Multi-Party Computation**: Compute on encrypted data
5. **Synthetic Data Generation**: Use AI to create realistic but fake data

In [ ]:
# Demonstrate aggregation as a privacy-preserving technique

# Original view of salary by medical condition could expose individuals
print("Individual-level data (potentially privacy-invasive):")
display(personal_data[['MedicalCondition', 'Salary']].head(10))

# Aggregated view provides insights without exposing individuals
agg_data = personal_data.groupby('MedicalCondition').agg({
    'Salary': ['mean', 'median', 'std', 'count'],
    'Age': ['mean', 'median', 'min', 'max'],
    'CreditScore': ['mean', 'median']
})

print("\nAggregated data (privacy-preserving):")
display(agg_data)

# Visualize aggregate data
plt.figure(figsize=(12, 5))
sns.barplot(x=personal_data['MedicalCondition'], y=personal_data['Salary'])
plt.title('Average Salary by Medical Condition')
plt.xlabel('Medical Condition')
plt.ylabel('Average Salary')
plt.xticks(rotation=45)
plt.show()

# Basic demonstration of differential privacy concept (simplified)
def add_laplacian_noise(data, column, sensitivity, epsilon=1.0):
    """Add Laplacian noise to achieve differential privacy."""
    scale = sensitivity / epsilon
    noise = np.random.laplace(0, scale, len(data))
    data_with_noise = data.copy()
    data_with_noise[column] = data_with_noise[column] + noise
    return data_with_noise

# Apply differential privacy to salary (simplified demonstration)
dp_data = add_laplacian_noise(personal_data, 'Salary', sensitivity=10000, epsilon=0.1)

# Compare original and differentially private salary statistics
comparison = pd.DataFrame({
    'Original': personal_data['Salary'].describe(),
    'With Differential Privacy': dp_data['Salary'].describe()
})
display(comparison)

# Visualize difference
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(personal_data['Salary'], kde=True)
plt.title('Original Salary Distribution')
plt.xlabel('Salary')

plt.subplot(1, 2, 2)
sns.histplot(dp_data['Salary'], kde=True)
plt.title('Salary with Differential Privacy')
plt.xlabel('Salary')

plt.tight_layout()
plt.show()

## 9. Ethical Data Collection Principles

Before collecting data, consider these ethical principles:

1. **Informed Consent**: Clearly explain how data will be used
2. **Purpose Limitation**: Only collect data needed for specific purposes
3. **Data Minimization**: Collect only what's necessary
4. **Transparency**: Be open about collection and usage practices
5. **Opt-out Options**: Allow individuals to refuse or withdraw consent

In [ ]:
# Create an example consent form template
consent_form = """
# DATA COLLECTION CONSENT FORM

## PURPOSE OF DATA COLLECTION
[Describe clearly why you're collecting the data]

## DATA BEING COLLECTED
- Personal identifiers (name, email)
- Demographic information (age, location)
- [List other data points]

## HOW DATA WILL BE USED
[Explain all intended uses of the data]

## DATA SHARING PRACTICES
[Explain if/how data will be shared with third parties]

## DATA SECURITY MEASURES
[Describe how data will be protected]

## DATA RETENTION PERIOD
Data will be retained for [time period] and then [disposal method].

## YOUR RIGHTS
- You can request access to your data
- You can request correction of inaccurate data
- You can request deletion of your data
- You can withdraw consent at any time

## CONSENT
[ ] I have read and understood this form
[ ] I consent to the collection and use of my data as described
"""

# Display the consent form with styling
display(HTML("<div style='background-color:#f8f9fa; padding:15px; border:1px solid #ddd; border-radius:5px'>" + 
             consent_form.replace('\n', '<br>') + "</div>"))

# Example data collection practices checklist
ethical_checklist = pd.DataFrame({
    'Practice': [
        'Obtain informed consent before collecting data',
        'Clearly state the purpose of data collection',
        'Collect only necessary data',
        'Provide opt-out options',
        'Secure data with appropriate measures',
        'Have a clear retention and deletion policy',
        'Allow individuals to access their data',
        'Be transparent about data sharing practices',
        'Consider impact on vulnerable populations',
        'Document data governance procedures'
    ],
    'Status': ['Implemented', 'Implemented', 'Needs Review', 'Implemented', 
              'In Progress', 'Not Started', 'Implemented', 'Needs Review',
              'Not Started', 'In Progress']
})

# Display the checklist with conditional formatting
def color_status(val):
    colors = {
        'Implemented': 'lightgreen',
        'In Progress': 'khaki',
        'Needs Review': 'lightsalmon',
        'Not Started': 'lightcoral'
    }
    return f'background-color: {colors.get(val, "white")}'

display(ethical_checklist.style.applymap(color_status, subset=['Status'])
       .set_caption("Ethical Data Collection Checklist"))

## 10. Bias and Fairness in Data Analysis

Algorithmic bias can lead to unfair or discriminatory outcomes. Let's explore how to detect and mitigate bias in data:

In [ ]:
# Create a sample dataset with potential bias
biased_data = pd.DataFrame({
    'Gender': np.random.choice(['Male', 'Female'], n_samples),
    'Age': ages,
    'Experience': np.random.randint(0, 30, n_samples),
    'Salary': salaries
})

# Deliberately introduce a bias - males tend to have higher salaries for the same experience
biased_data.loc[biased_data['Gender'] == 'Male', 'Salary'] *= 1.15

# Analyze the dataset for gender pay disparities
gender_pay_gap = biased_data.groupby('Gender')['Salary'].agg(['mean', 'median'])
print("Pay by gender:")
display(gender_pay_gap)

# Calculate pay gap percentage
pay_gap_pct = ((gender_pay_gap.loc['Male', 'mean'] - gender_pay_gap.loc['Female', 'mean']) / 
               gender_pay_gap.loc['Male', 'mean'] * 100)
print(f"Gender pay gap: {pay_gap_pct:.2f}%")

# Visualize the bias
plt.figure(figsize=(14, 6))

# Salary distribution by gender
plt.subplot(1, 2, 1)
sns.histplot(data=biased_data, x='Salary', hue='Gender', element='step', kde=True)
plt.title('Salary Distribution by Gender')
plt.xlabel('Salary')

# Salary vs. Experience by gender
plt.subplot(1, 2, 2)
sns.scatterplot(data=biased_data, x='Experience', y='Salary', hue='Gender')
plt.title('Salary vs. Experience by Gender')
plt.xlabel('Years of Experience')
plt.ylabel('Salary')

plt.tight_layout()
plt.show()

# Examining the relationship with a regression plot
plt.figure(figsize=(10, 6))
sns.lmplot(data=biased_data, x='Experience', y='Salary', hue='Gender', height=6, aspect=1.5)
plt.title('Salary vs Experience by Gender with Regression Lines')
plt.show()

# Create a simple function to check for bias in a dataset
def check_for_bias(data, feature, target):
    """
    Check for potential bias in a dataset by comparing distributions and statistics
    across different groups defined by a categorical feature.
    
    Parameters:
    data (pandas.DataFrame): The dataset to check
    feature (str): The categorical feature to check for bias (e.g., 'Gender', 'Race')
    target (str): The target variable that might be affected by bias (e.g., 'Salary')
    """
    # Get unique values in the feature
    groups = data[feature].unique()
    
    # Calculate statistics for each group
    stats = data.groupby(feature)[target].agg(['count', 'mean', 'median', 'std'])
    
    # Calculate the overall stats
    overall_mean = data[target].mean()
    
    # Calculate the deviation from the mean
    stats['deviation'] = ((stats['mean'] - overall_mean) / overall_mean * 100).round(2)
    
    print(f"Bias check for {target} across {feature} groups:")
    display(stats)
    
    # Simple statistical test (ANOVA)
    from scipy import stats as scipy_stats
    groups_data = [data[data[feature] == group][target] for group in groups]
    f_stat, p_value = scipy_stats.f_oneway(*groups_data)
    
    print(f"ANOVA F-statistic: {f_stat:.4f}")
    print(f"p-value: {p_value:.4f}")
    
    if p_value < 0.05:
        print(f"There is a statistically significant difference in {target} across {feature} groups.")
    else:
        print(f"No statistically significant difference detected in {target} across {feature} groups.")
    
    # Visualize the distributions
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=feature, y=target, data=data)
    plt.title(f'{target} by {feature}')
    plt.show()

# Apply the bias check
check_for_bias(biased_data, 'Gender', 'Salary')

## 11. Ethical Guidelines for Data Scientists

As data scientists, we should follow these ethical guidelines:

1. **Do No Harm**: Ensure your work doesn't cause harm to individuals or communities
2. **Respect Privacy**: Protect personal information and respect privacy rights
3. **Be Transparent**: Be open about methods, limitations, and potential biases
4. **Ensure Fairness**: Work to identify and mitigate bias in data and algorithms
5. **Take Responsibility**: Accept accountability for the outcomes of your work
6. **Consider Long-term Impacts**: Evaluate potential long-term consequences
7. **Protect Vulnerable Populations**: Take extra care with data from vulnerable groups

## 12. Case Studies in Data Ethics

Let's examine some real-world data ethics scenarios and discuss appropriate responses:

### Case Study 1: The Cambridge Analytica Scandal
Cambridge Analytica collected data from millions of Facebook users without proper consent for political targeting.

**Lessons learned:**
- Always obtain informed consent
- Be transparent about data usage
- Have clear data sharing policies
- Implement proper access controls

### Case Study 2: Facial Recognition and Bias
Several facial recognition systems have shown significant bias in identifying people of certain demographics.

**Lessons learned:**
- Test systems across diverse populations
- Monitor for disparate impact
- Continuously update training data
- Consider withholding technology when harm outweighs benefits

### Case Study 3: COVID-19 Contact Tracing
The pandemic highlighted tensions between public health needs and individual privacy concerns.

**Lessons learned:**
- Use privacy-by-design principles
- Implement data minimization
- Set clear data retention limits
- Provide transparency about usage

## 13. Data Ethics Assessment Framework

When starting a new data project, consider this ethical assessment framework:

In [ ]:
# Create a data ethics assessment framework
ethics_assessment = pd.DataFrame({
    'Question': [
        'Is informed consent obtained from data subjects?',
        'Is data collection limited to necessary information?',
        'Are data security measures implemented?',
        'Is data anonymized when possible?',
        'Are there procedures for access, correction, and deletion requests?',
        'Is the system tested for bias across different demographic groups?',
        'Are there mechanisms to explain algorithm decisions?',
        'Has a privacy impact assessment been conducted?',
        'Are there processes to handle potential harms?',
        'Is there a defined data retention and deletion policy?'
    ],
    'Category': [
        'Consent', 'Data Minimization', 'Security', 'Privacy', 'Individual Rights',
        'Fairness', 'Transparency', 'Impact Assessment', 'Harm Prevention', 'Retention'
    ],
    'Yes/No': ['', '', '', '', '', '', '', '', '', ''],
    'Notes': ['', '', '', '', '', '', '', '', '', '']
})

# Display the assessment framework as a form
display(ethics_assessment.style.set_caption("Data Ethics Assessment Framework"))

# Sample ethical decision-making process
ethical_decision_flow = """
1. **Identify Ethical Concerns**
   - What are the potential privacy impacts?
   - Could this create or reinforce bias?
   - Are vulnerable populations at risk?

2. **Consult Stakeholders**
   - Data subjects
   - Domain experts
   - Ethics specialists
   - Legal advisors

3. **Analyze Alternatives**
   - Can we achieve the same goal with less sensitive data?
   - Are there more privacy-preserving methods?
   - What are the tradeoffs of different approaches?

4. **Implement Safeguards**
   - Technical controls (encryption, access control)
   - Policy controls (usage limitations, review processes)
   - Training and awareness

5. **Monitor Outcomes**
   - Audit data usage
   - Evaluate impacts
   - Adjust as needed
"""

# Display the decision-making process
display(HTML("<div style='background-color:#f8f9fa; padding:15px; border:1px solid #ddd; border-radius:5px'>" + 
             ethical_decision_flow.replace('\n', '<br>') + "</div>"))

## 14. Conclusion and Best Practices

### Key Takeaways:

1. **Privacy and ethics are fundamental** to responsible data science practice
2. **De-identification techniques** help protect individual privacy while enabling analysis
3. **Bias awareness and mitigation** are essential for fair outcomes
4. **Legal compliance** with regulations like GDPR is mandatory in many contexts
5. **Ethical frameworks** can guide decision-making for complex data issues

### Best Practices:

1. **Privacy by Design**: Build privacy into projects from the start
2. **Data Minimization**: Collect only what you need
3. **Regular Audits**: Review data practices and outcomes
4. **Transparency**: Be open about methods and limitations
5. **Ethical Review**: Consider ethical implications before starting projects

## 15. Additional Resources

For more information on data ethics and privacy, explore these resources:

1. [Data Ethics Framework (UK Government)](https://www.gov.uk/government/publications/data-ethics-framework)
2. [Association for Computing Machinery (ACM) Code of Ethics](https://www.acm.org/code-of-ethics)
3. [Data Science Association Code of Professional Conduct](https://www.datascienceassn.org/code-of-conduct.html)
4. Book: "Weapons of Math Destruction" by Cathy O'Neil
5. Book: "Privacy in the Age of Big Data" by Theresa Payton and Ted Claypoole

## 16. Exercises

1. Identify potential ethical concerns in a dataset of your choice
2. Apply de-identification techniques to a sample dataset
3. Create a privacy policy for a fictional data science project
4. Conduct a bias assessment on an algorithm or dataset
5. Design a consent form for a specific data collection scenario